<a href="https://colab.research.google.com/github/vishnu062006/bis-lab/blob/main/ANT_COLONY_OPTIMIZATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import random

random.seed(1)

# Distance matrix: city 0 is the depot
d = [
    [0, 2, 9, 10, 7, 3],
    [2, 0, 6, 4, 3, 5],
    [9, 6, 0, 8, 5, 7],
    [10, 4, 8, 0, 6, 4],
    [7, 3, 5, 6, 0, 2],
    [3, 5, 7, 4, 2, 0]
]

demand = [0, 2, 3, 4, 2, 3]
n = len(d)
capacity = 7
vehicles = 3
ants = 10
iterations = 50
alpha = 1
beta = 2
rho = 0.5
Q = 100

pher = [[1.0 if i != j else 0 for j in range(n)]
        for i in range(n)]

best_routes = None
best_dist = float('inf')

for _ in range(iterations):
    solutions = []

    for _ in range(ants):
        unvisited = set(range(1, n))
        routes = []
        total = 0

        for v in range(vehicles):
            route = [0]
            load = 0

            while True:
                i = route[-1]
                choices = [j for j in unvisited
                           if demand[j] <= capacity - load]

                if not choices:
                    break

                weights = [
                    pher[i][j] ** alpha * (1 / d[i][j]) ** beta
                    for j in choices
                ]

                r = random.random() * sum(weights)
                s = 0

                for j, w in zip(choices, weights):
                    s += w
                    if s >= r:
                        route.append(j)
                        load += demand[j]
                        unvisited.remove(j)
                        break

            if len(route) > 1:
                route.append(0)
                total += sum(d[route[k]][route[k + 1]]
                             for k in range(len(route) - 1))
                routes.append(route)

            if not unvisited:
                break

        if not unvisited:
            solutions.append((routes, total))

            if total < best_dist:
                best_dist = total
                best_routes = [r[:] for r in routes]

    # Pheromone evaporation
    for i in range(n):
        for j in range(n):
            if i != j:
                pher[i][j] *= (1 - rho)

    # Pheromone deposit
    for routes, length in solutions:
        for route in routes:
            for k in range(len(route) - 1):
                a, b = route[k], route[k + 1]
                pher[a][b] += Q / length
                pher[b][a] += Q / length

print("Best Vehicle Routes:")
for i, route in enumerate(best_routes, 1):
    print("Vehicle", i, ":", " -> ".join(map(str, route)))
print("Minimum Total Distance:", best_dist)

Best Vehicle Routes:
Vehicle 1 : 0 -> 1 -> 4 -> 2 -> 0
Vehicle 2 : 0 -> 5 -> 3 -> 0
Minimum Total Distance: 36
